# The XOR Problem: Why One Perceptron Isn't Enough

## What is XOR?

XOR stands for "exclusive or." It outputs 1 when exactly one of the two inputs is 1, and 0 otherwise:

| Input 1 | Input 2 | XOR output |
|---------|---------|------------|
| 0       | 0       | 0          |
| 0       | 1       | 1          |
| 1       | 0       | 1          |
| 1       | 1       | 0          |

This seems simple enough — but it turns out to be a fundamental challenge for a single perceptron.

## Why a Single Perceptron Fails at XOR

Picture the four input pairs as points on a grid. The two cases that should output 1 — [0,1] and [1,0] — are in opposite corners. The two cases that should output 0 — [0,0] and [1,1] — are in the other two corners. The points form an X shape, with the two classes checkerboarded.

A perceptron's decision boundary is always a straight line. No matter how you tilt or shift that line, you cannot place the two 1s on one side and the two 0s on the other. Try it — there's simply no straight line that works. This is what "not linearly separable" means in practice.

We proved this experimentally in the previous notebook: even after 5,000 epochs, the perceptron couldn't learn XOR because it was searching for something that doesn't exist.

## How a Hidden Layer Solves It

The insight is to stop trying to solve XOR directly, and instead build intermediate features that *are* linearly separable.

Here's the trick: XOR is logically equivalent to "OR but not AND." So if we build two intermediate features — one that computes OR and one that computes AND — and then combine them, we can compute XOR.

Those intermediate features live in a **hidden layer**: a layer of perceptrons that the outside world never sees directly. Their outputs feed into one final perceptron that combines them.

This is the core motivation for multi-layer networks. By stacking layers, each layer can learn to build more complex features out of simpler ones, until the final layer can solve problems that no single straight line could handle.

In the code below, we let scikit-learn build and train a two-layer network (one hidden layer with 2 nodes) on the XOR problem and see that it succeeds where a single perceptron couldn't.

In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier

## Setting Up the XOR Dataset

Same four input combinations as before, with XOR labels.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

## Visualizing the Problem

Before building the network, let's see exactly why a single perceptron can't solve XOR — and why that forces us to add a hidden layer.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: XOR points on the grid
ax1 = axes[0]
for lbl in [0, 1]:
    mask = y == lbl
    ax1.scatter(X[mask, 0], X[mask, 1], s=280,
                c='steelblue' if lbl == 1 else 'salmon',
                edgecolors='k', zorder=3, linewidths=1.5, label=f'XOR = {lbl}')
for xi, yi in zip(X, y):
    ax1.annotate(f'({int(xi[0])},{int(xi[1])})', xi,
                 textcoords='offset points', xytext=(8, 5), fontsize=10)

x_line = np.linspace(-0.3, 1.3, 100)
for slope, intercept, ls in [(-1, 0.5, '--'), (0, 0.5, ':'), (1, -0.3, '-.')]:
    ax1.plot(x_line, slope * x_line + intercept,
             color='gray', linewidth=1.2, linestyle=ls, alpha=0.5)

ax1.set_xlim(-0.4, 1.4); ax1.set_ylim(-0.4, 1.4)
ax1.set_xlabel('x1'); ax1.set_ylabel('x2')
ax1.set_title('XOR points — try any line,\nnone can separate blue from red')
ax1.legend()

# Right: explanation panel
ax2 = axes[1]
ax2.axis('off')
ax2.text(0.5, 0.92, 'Why no line works', ha='center', va='top',
         fontsize=13, fontweight='bold', transform=ax2.transAxes)
ax2.text(0.5, 0.78,
         'XOR=0:  (0,0) and (1,1)  — same diagonal\nXOR=1:  (0,1) and (1,0) — other diagonal',
         ha='center', va='top', fontsize=10, family='monospace', transform=ax2.transAxes)
ax2.text(0.5, 0.56,
         'Any line that isolates one XOR=0\ncorner also captures a XOR=1 corner.\nThe two classes are interleaved.',
         ha='center', va='top', fontsize=10, color='#555', transform=ax2.transAxes)
ax2.text(0.5, 0.32, 'Not linearly separable.',
         ha='center', va='top', fontsize=12, color='darkred', fontweight='bold',
         transform=ax2.transAxes,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff3f3', edgecolor='darkred'))
ax2.text(0.5, 0.12,
         'Fix: add a hidden layer to build\nnew features that ARE separable.',
         ha='center', va='top', fontsize=10, color='steelblue', transform=ax2.transAxes,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f6ff', edgecolor='steelblue'))

plt.tight_layout()
plt.show()

## Building a Multi-Layer Network

The key setting here is `hidden_layer_sizes=(2,)`. This creates a network with one hidden layer containing 2 nodes — exactly the "OR node" and "AND node" construction described above, but learned automatically from data rather than hand-coded.

We use `tanh` as the activation function (instead of the simple 0/1 step function). Tanh is a smooth, differentiable curve that outputs values between -1 and 1. Being smooth (differentiable) matters because the training algorithm — backpropagation — needs to compute gradients, which require derivatives. A sharp 0/1 step has no useful derivative, so it can't be trained effectively with gradient-based methods.

The `learning_rate_init=0.1` controls how big each weight adjustment is during training, and `max_iter=500` is the maximum number of passes through the data.

In [ ]:
nn = MLPClassifier(hidden_layer_sizes = (2,),
                   learning_rate_init = 0.1,
                   max_iter = 500,
                   activation = 'tanh',
                   random_state = 1)

## Training and Evaluating

We train the network and immediately check its predictions on the same four inputs. Unlike the single perceptron, this network should get all four right.

In [ ]:
nn.fit(X,y)
predicts = nn.predict(X)
print(predicts)

## Inspecting the Learned Weights

We can look at what the network actually learned. `nn.coefs_` contains the weight matrices for each layer — the first matrix connects the inputs to the hidden layer, and the second connects the hidden layer to the output. `nn.intercepts_` contains the bias terms for each layer.

These numbers won't be as clean as the hand-crafted OR/AND weights from the course notes (the network found its own equivalent solution), but you can verify that the hidden layer nodes are doing something analogous to OR and NOT-AND.

In [ ]:
print(nn.coefs_)
print('\n')
print(nn.intercepts_)

## What the Hidden Layer Learns

The network solved XOR by learning two hidden nodes that transform the four input points into a new 2D space — one where the two classes *are* linearly separable.

The plot on the left shows the original input space: no line can separate the XOR=0 points from the XOR=1 points. The plot on the right shows where those same four points land after passing through the hidden layer. The output layer then draws a single line to separate them.

This is the core power of hidden layers: they don't just classify directly — they *re-represent* the input in a form where classification becomes easier.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Hidden layer activations: Z = tanh(X @ W + b)
Z = np.tanh(X @ nn.coefs_[0] + nn.intercepts_[0])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
labels_str = [f'({int(xi[0])},{int(xi[1])})' for xi in X]

# Left: original input space
ax1 = axes[0]
for lbl in [0, 1]:
    mask = y == lbl
    ax1.scatter(X[mask, 0], X[mask, 1], s=250,
                c='steelblue' if lbl == 1 else 'salmon',
                edgecolors='k', zorder=3, label=f'XOR = {lbl}')
for xi, s in zip(X, labels_str):
    ax1.annotate(s, xi, textcoords='offset points', xytext=(8, 5), fontsize=10)
ax1.set_xlim(-0.5, 1.5); ax1.set_ylim(-0.5, 1.5)
ax1.set_xlabel('x1 (original)'); ax1.set_ylabel('x2 (original)')
ax1.set_title('Input space — not linearly separable')
ax1.legend()

# Right: hidden layer space
ax2 = axes[1]
for lbl in [0, 1]:
    mask = y == lbl
    ax2.scatter(Z[mask, 0], Z[mask, 1], s=250,
                c='steelblue' if lbl == 1 else 'salmon',
                edgecolors='k', zorder=3, label=f'XOR = {lbl}')
for zi, s in zip(Z, labels_str):
    ax2.annotate(s, zi, textcoords='offset points', xytext=(8, 5), fontsize=10)

# Draw the separating line learned by the output layer
w_out = nn.coefs_[1].flatten()
b_out = nn.intercepts_[1][0]
z0_range = np.linspace(Z[:, 0].min() - 0.2, Z[:, 0].max() + 0.2, 100)
if abs(w_out[1]) > 1e-6:
    z1_line = -(w_out[0] * z0_range + b_out) / w_out[1]
    ax2.plot(z0_range, z1_line, 'k--', linewidth=1.5, label='Output layer boundary')

ax2.set_xlabel('Hidden node z1'); ax2.set_ylabel('Hidden node z2')
ax2.set_title('Hidden layer space — now linearly separable')
ax2.legend()

fig.suptitle('The hidden layer transforms the input into a space where XOR can be solved.',
             fontsize=11)
plt.tight_layout()
plt.show()